# Jupyter Baseline Models: Boro Rice Production

This notebook uses **Table V: Production of Major Agricultural Commodities** from the supplied Excel workbook.

**Target:** Boro Rice Production  
**Models:** Linear Regression, Decision Tree Regressor, and K-Nearest Neighbors (KNN) Regressor

The implementation intentionally uses basic preprocessing, exploratory data analysis (EDA), and baseline models only.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor

from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


## 2. Load Dataset

In [ ]:
file_path = "/mnt/data/Apr26 (For Shounok 16.07.2026).xlsx"

data = pd.read_excel(file_path, sheet_name="Table V", header=None)

data.head(10)


## 3. Understand the Dataset

In [ ]:
print("Dataset shape:", data.shape)

print("\nFirst 15 rows:")
display(data.head(15))


The sheet contains agricultural commodities such as Aus Rice, Aman Rice, Boro Rice, Wheat, Sugar Cane, and Rape & Mustard. The original Excel sheet contains header rows, so it needs cleaning before machine learning.


## 4. Select the Actual Data

In [ ]:
df = data.iloc[6:, :13].copy()

df.columns = [
    "Period",
    "Aus_Production",
    "Aus_Area",
    "Aman_Production",
    "Aman_Area",
    "Boro_Production",
    "Boro_Area",
    "Wheat_Production",
    "Wheat_Area",
    "Sugarcane_Production",
    "Sugarcane_Area",
    "RapeMustard_Production",
    "RapeMustard_Area"
]

df.head()


## 5. Remove Unnecessary Rows

In [ ]:
df = df[
    df["Period"].astype(str).str.strip().str.match(r"^\d{4}-\d{2}$")
].copy()

df.reset_index(drop=True, inplace=True)

print("Rows after cleaning:", len(df))

display(df.head())


## 6. Convert Numerical Columns

In [ ]:
numeric_columns = df.columns[1:]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df.head()


## 7. Basic Dataset Information

In [ ]:
print("Dataset Information:")
df.info()

print("\nDataset Shape:")
print(df.shape)


## 8. Check Missing Values

In [ ]:
print("Missing values in each column:")
print(df.isnull().sum())


## 9. Handle Missing Values

In [ ]:
df = df.dropna()

print("Shape after removing missing values:", df.shape)

print("\nMissing values:")
print(df.isnull().sum())


## 10. Descriptive Statistics

In [ ]:
df.describe()


# Exploratory Data Analysis (EDA)

## 11. Boro Rice Production Over Time

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    df["Period"],
    df["Boro_Production"],
    marker="o"
)

plt.title("Boro Rice Production Over Time")
plt.xlabel("Year")
plt.ylabel("Production ('000' Metric Tons)")

plt.xticks(rotation=45)

plt.grid()
plt.show()


## 12. Distribution of Boro Production

In [ ]:
plt.figure(figsize=(8, 5))

plt.hist(
    df["Boro_Production"],
    bins=10
)

plt.title("Distribution of Boro Rice Production")
plt.xlabel("Boro Production")
plt.ylabel("Frequency")

plt.show()


## 13. Boxplot of Boro Production

In [ ]:
plt.figure(figsize=(8, 5))

plt.boxplot(df["Boro_Production"])

plt.title("Boxplot of Boro Rice Production")
plt.ylabel("Production ('000' Metric Tons)")

plt.show()


## 14. Correlation Matrix

In [ ]:
plt.figure(figsize=(12, 8))

correlation = df.drop(columns=["Period"]).corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Matrix")

plt.show()


## 15. Boro Area vs Boro Production

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["Boro_Area"],
    df["Boro_Production"]
)

plt.title("Boro Area vs Boro Production")
plt.xlabel("Boro Area ('000' acres)")
plt.ylabel("Boro Production ('000' Metric Tons)")

plt.grid()
plt.show()


## 16. Aman Production vs Boro Production

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["Aman_Production"],
    df["Boro_Production"]
)

plt.title("Aman Production vs Boro Production")
plt.xlabel("Aman Production")
plt.ylabel("Boro Production")

plt.grid()
plt.show()


## 17. Aus Production vs Boro Production

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    df["Aus_Production"],
    df["Boro_Production"]
)

plt.title("Aus Production vs Boro Production")
plt.xlabel("Aus Production")
plt.ylabel("Boro Production")

plt.grid()
plt.show()


# Data Preprocessing

## 18. Select Features and Target

The target is **Boro Rice Production**. It is not included as an input feature.


In [ ]:
X = df[
    [
        "Aus_Production",
        "Aus_Area",
        "Aman_Production",
        "Aman_Area",
        "Wheat_Production",
        "Wheat_Area",
        "Sugarcane_Production",
        "Sugarcane_Area",
        "RapeMustard_Production",
        "RapeMustard_Area"
    ]
]

y = df["Boro_Production"]

print("Features:")
display(X.head())

print("\nTarget:")
display(y.head())


## 19. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)


This uses 80% of the data for training and 20% for testing.

## 20. Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)


Scaling is especially useful for KNN because KNN calculates distances between data points.

# Baseline Model 1: Linear Regression

## 21. Train Linear Regression

In [ ]:
linear_model = LinearRegression()

linear_model.fit(
    X_train,
    y_train
)

linear_prediction = linear_model.predict(X_test)


## 22. Evaluate Linear Regression

In [ ]:
linear_mae = mean_absolute_error(
    y_test,
    linear_prediction
)

linear_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        linear_prediction
    )
)

linear_r2 = r2_score(
    y_test,
    linear_prediction
)

print("Linear Regression")
print("----------------------")
print("MAE:", linear_mae)
print("RMSE:", linear_rmse)
print("R2 Score:", linear_r2)


# Baseline Model 2: Decision Tree Regressor

## 23. Train Decision Tree

In [ ]:
tree_model = DecisionTreeRegressor(
    random_state=42
)

tree_model.fit(
    X_train,
    y_train
)

tree_prediction = tree_model.predict(X_test)


## 24. Evaluate Decision Tree

In [ ]:
tree_mae = mean_absolute_error(
    y_test,
    tree_prediction
)

tree_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        tree_prediction
    )
)

tree_r2 = r2_score(
    y_test,
    tree_prediction
)

print("Decision Tree")
print("----------------------")
print("MAE:", tree_mae)
print("RMSE:", tree_rmse)
print("R2 Score:", tree_r2)


# Baseline Model 3: K-Nearest Neighbors (KNN) Regressor

## 25. Train KNN

In [ ]:
knn_model = KNeighborsRegressor(
    n_neighbors=3
)

knn_model.fit(
    X_train_scaled,
    y_train
)

knn_prediction = knn_model.predict(
    X_test_scaled
)


## 26. Evaluate KNN

In [ ]:
knn_mae = mean_absolute_error(
    y_test,
    knn_prediction
)

knn_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        knn_prediction
    )
)

knn_r2 = r2_score(
    y_test,
    knn_prediction
)

print("KNN Regression")
print("----------------------")
print("MAE:", knn_mae)
print("RMSE:", knn_rmse)
print("R2 Score:", knn_r2)


# Model Comparison

## 27. Create a Model Comparison Table

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "KNN"
    ],

    "MAE": [
        linear_mae,
        tree_mae,
        knn_mae
    ],

    "RMSE": [
        linear_rmse,
        tree_rmse,
        knn_rmse
    ],

    "R2 Score": [
        linear_r2,
        tree_r2,
        knn_r2
    ]
})

results


## 28. Compare Model Performance Visually

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    results["Model"],
    results["R2 Score"]
)

plt.title("Comparison of R2 Scores")
plt.xlabel("Model")
plt.ylabel("R2 Score")

plt.xticks(rotation=20)

plt.show()


## 29. Actual vs Predicted Boro Production

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    y_test,
    linear_prediction,
    label="Linear Regression"
)

plt.scatter(
    y_test,
    tree_prediction,
    label="Decision Tree"
)

plt.scatter(
    y_test,
    knn_prediction,
    label="KNN"
)

plt.xlabel("Actual Boro Production")
plt.ylabel("Predicted Boro Production")

plt.title("Actual vs Predicted Boro Production")

plt.legend()
plt.grid()

plt.show()


## 30. Final Comparison

In [ ]:
print("Final Model Performance")
print("========================")

for i in range(len(results)):
    print("\nModel:", results.loc[i, "Model"])
    print("MAE:", round(results.loc[i, "MAE"], 2))
    print("RMSE:", round(results.loc[i, "RMSE"], 2))
    print("R2 Score:", round(results.loc[i, "R2 Score"], 2))


## 31. Find the Best Baseline Model

In [ ]:
best_model = results.loc[
    results["R2 Score"].idxmax()
]

print("Best Baseline Model:")
print(best_model["Model"])

print("\nR2 Score:", best_model["R2 Score"])


# Deliverables Covered

- EDA with visualizations
- Dataset preprocessing and missing-value handling
- Train-test split and feature scaling
- Linear Regression baseline model
- Decision Tree Regressor baseline model
- KNN Regressor baseline model
- MAE, RMSE, and R² evaluation with a comparison table

The workbook has a small number of yearly observations, so the results should be treated as baseline benchmarks.
